# Context-Aware Trace Debugging with Falcon AI: From Error to Fix in Minutes

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/future-agi/cookbooks/blob/cookbook/quickstart-notebooks/use-cases/falcon-ai-context-aware-debugging.ipynb)
[![View on GitHub](https://img.shields.io/badge/View_on_GitHub-181717?logo=github&logoColor=white)](https://github.com/future-agi/cookbooks/blob/cookbook/quickstart-notebooks/use-cases/falcon-ai-context-aware-debugging.ipynb)

| Time | Difficulty |
|------|------------|
| 15 min | Beginner |

You launched a research assistant for your team last week. It searches your internal paper database and synthesizes summaries with citations. This morning a colleague pings you: "this paper you cited doesn't exist." You open the dashboard, find the trace, and there it is. A confidently-formatted citation that the model invented because the search tool returned no results for the topic.

You do not need a full regression suite right now. You do not need to build a dataset or run a sweep. You need to know what broke in this one trace and what to change so it stops happening. Fast.

The slow version of this is familiar: open the trace, expand the span tree, scroll through 6 nested spans, copy the system prompt out, copy the model output out, diff them in your head, write the fix in a notebook, run it, look at the new trace, repeat if it didn't work. Easily an hour for one trace, longer if you context-switch.

This is the failure mode that got a New York lawyer sanctioned in 2023 for citing six cases that ChatGPT had completely fabricated. The pattern is the same: a search returns nothing, the model fills the gap with plausible-looking output, and a downstream user trusts it.

This notebook walks the fast version, powered by FutureAGI's **Tracing** + **Falcon AI** stack working together. You open Falcon AI directly on the failing trace, and the chat input shows the trace as a context chip automatically. No copy-pasting trace IDs, no re-establishing "which trace are we talking about" between turns. You ask one open question, drill into the span with `/analyze-trace-errors`, get a verbatim prompt fix from `/fix-with-falcon`, paste it into your code, re-run the same query, watch the agent refuse instead of fabricate. End to end in under 15 minutes.

**Prerequisites:**
- FutureAGI account: [app.futureagi.com](https://app.futureagi.com)
- API keys: `FI_API_KEY` and `FI_SECRET_KEY` (see [Get your API keys](https://docs.futureagi.com/docs/admin-settings))
- OpenAI API key (`OPENAI_API_KEY`)
- Python 3.10+

## Install

The simplest way to run this notebook is in **Google Colab** (click the badge at the top). Colab has Python 3.11 and the `%pip install` cell below works out of the box.

If you're running locally, you need Python 3.10+ (`fi-instrumentation-otel` won't import on 3.9).

In [ ]:
%pip install fi-instrumentation-otel traceai-openai openai

In [ ]:
import os
os.environ["FI_API_KEY"] = "your-fi-api-key"
os.environ["FI_SECRET_KEY"] = "your-fi-secret-key"
os.environ["OPENAI_API_KEY"] = "your-openai-key"

## Step 1: Build a research assistant with a small knowledge base

The agent has one tool, `search_papers`, that returns hits from a tiny three-paper mock database. The system prompt is intentionally permissive: it tells the model to answer with citations, but it does not say what to do when the search comes back empty. That gap is where the failure lives.

In [ ]:
import json
from openai import OpenAI

client = OpenAI()

SYSTEM_PROMPT = """You are a research assistant for an ML research team.
Answer questions using the search_papers tool. Provide citations to support your claims."""

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_papers",
            "description": "Search the team's internal database of ML papers",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Topic or keyword to search for"},
                },
                "required": ["query"],
            },
        },
    },
]


def search_papers(query: str) -> dict:
    db = {
        "transformer": [
            {"title": "Attention Is All You Need", "authors": "Vaswani et al.", "year": 2017, "venue": "NeurIPS"},
        ],
        "diffusion": [
            {"title": "Denoising Diffusion Probabilistic Models", "authors": "Ho et al.", "year": 2020, "venue": "NeurIPS"},
        ],
        "rlhf": [
            {"title": "Training language models to follow instructions with human feedback", "authors": "Ouyang et al.", "year": 2022, "venue": "NeurIPS"},
        ],
    }
    q = query.lower()
    for keyword, papers in db.items():
        if keyword in q:
            return {"results": papers, "total": len(papers)}
    return {"results": [], "total": 0}


TOOL_MAP = {"search_papers": search_papers}


def handle_message(messages: list) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": SYSTEM_PROMPT}] + messages,
        tools=TOOLS,
    )
    msg = response.choices[0].message

    if msg.tool_calls:
        tool_messages = [msg]
        for tc in msg.tool_calls:
            result = TOOL_MAP[tc.function.name](**json.loads(tc.function.arguments))
            tool_messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result),
            })
        followup = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "system", "content": SYSTEM_PROMPT}] + messages + tool_messages,
            tools=TOOLS,
        )
        return followup.choices[0].message.content

    return msg.content

Three topics covered (transformers, diffusion, RLHF). Anything else hits the empty-result branch.

## Step 2: Add tracing so Falcon AI can read the spans

In [ ]:
from fi_instrumentation import register, FITracer, using_user, using_session
from fi_instrumentation.fi_types import ProjectType
from traceai_openai import OpenAIInstrumentor

trace_provider = register(
    project_type=ProjectType.OBSERVE,
    project_name="research-assistant-debug",
)
OpenAIInstrumentor().instrument(tracer_provider=trace_provider)
tracer = FITracer(trace_provider.get_tracer("research-assistant-debug"))


@tracer.agent(name="research_assistant")
def traced_handle(user_id: str, session_id: str, messages: list) -> str:
    with using_user(user_id), using_session(session_id):
        return handle_message(messages)

`@tracer.agent` makes the entire request show up as one parent span with the OpenAI calls and tool calls nested underneath. That nesting is what `/fix-with-falcon` needs in order to read the verbatim system prompt and model output later.

## Step 3: Trigger the failing trace

Two queries: one inside the knowledge base and one outside. The second one is the trace you'll debug.

In [ ]:
# In-database query: should work cleanly
traced_handle(
    user_id="alice",
    session_id="session-good",
    messages=[{"role": "user", "content": "What's the seminal paper on transformers?"}],
)

# Outside-database query: should expose the failure
answer = traced_handle(
    user_id="alice",
    session_id="session-bad",
    messages=[{"role": "user", "content": "What are the key papers on contrastive learning for self-supervised vision?"}],
)
print(answer)

trace_provider.force_flush()

Look closely at what the model returned for the second query. The agent likely acknowledged the empty database ("there are currently no papers available... However, I can provide you with general insights") and then named three specific papers (SimCLR, MoCo, BYOL) with one-line descriptions, framed as "key papers often referenced." The hedge phrasing makes the response sound careful, but the names and descriptions are not grounded in any tool result. The system prompt told the model to provide citations and never told it what to do when the tool returned nothing, so the model filled the gap from its training data and dressed it up as helpfulness.

Open **Tracing** in the dashboard, select `research-assistant-debug`, and click into the second trace. The span tree shows the empty tool result and the fabricated content side by side. That contradiction is the bug.

## Step 4: Debug the trace conversationally using page context

**This step is done in the dashboard, not the notebook.**

Open the failing trace in the **Tracing** Feed. Click into it so the trace detail page is the active view. Now press `Cmd+K` (Mac) or `Ctrl+K` (Windows) to open the Falcon AI sidebar.

Look at the chat input. There is a **context chip** above the message box showing the current trace ID (something like `trace 7ab8c…`). You did not type that. Falcon AI saw what page you were on and attached it to the conversation. Every question you ask in this chat will be answered against that specific trace until you remove or replace the chip.

> **Tip.** Without page context, you would have to start every question with "Look at trace 7ab8c… in project research-assistant-debug, …" and re-paste the ID for each follow-up. With it, you ask the question and Falcon AI already knows what you mean. This is the difference between debugging by chat and debugging by chat that knows what you are looking at.

You will run three turns in the same chat. Each turn builds on the previous one, and the trace context carries through automatically.

**Turn 1 (≈0:30 in): the open question.** Start by asking what went wrong, the way you would ask a teammate looking over your shoulder.

> What went wrong with this trace?

Falcon AI reads the trace summary and gives an exploratory diagnosis: the empty-result tool call, the fallback to general knowledge, and a list of likely root causes (data, retrieval, indexing).

![Falcon AI sidebar opened on the failing trace, with the trace context chip in the chat input and an exploratory diagnosis of the empty search result](https://fi-cookbook-assets.s3.ap-south-1.amazonaws.com/use-cases/falcon-ai-context-aware-debugging/turn-1-open-question.png)

Notice the angle. The first-turn response treats "what went wrong" as a question about the system as a whole and leads with the data and retrieval layer. That is a reasonable default; in production, an empty result on a famous topic is more often a retrieval bug than an agent behavior bug. For our case, the database is intentionally tiny (three papers), so the retrieval is working correctly and the real failure is the agent's response to an empty result. The next turn narrows to the agent.

**Turn 2 (≈2:00 in): drill into the span with `/analyze-trace-errors`.** In the same chat, type:

> /analyze-trace-errors

Falcon AI runs the analyze-trace-errors skill against the trace already in context. It calls `explore_trace_legacy` and `read_trace_span(exact=True)` on the LLM span, submits structured findings, and writes a quality scorecard.

Two things to notice. First, the skill captures both layers of the failure (retrieval returned nothing, then the agent hallucinated) instead of picking one. Second, the recommended prompt fix already previews the next turn's diff but as advice rather than a copy-pasteable change: that is the difference between `/analyze-trace-errors` (diagnosis with suggestions) and `/fix-with-falcon` (one concrete prompt change you can paste).

![Falcon AI showing the structured /analyze-trace-errors output with category findings, severity, and a quality scorecard for the same trace](https://fi-cookbook-assets.s3.ap-south-1.amazonaws.com/use-cases/falcon-ai-context-aware-debugging/turn-2-analyze-trace-errors.png)

**Turn 3 (≈4:00 in): get the prompt diff with `/fix-with-falcon`.** Type:

> /fix-with-falcon

Falcon AI runs the fix-with-falcon skill, reads the verbatim system prompt and model output one more time (it does not trust your description of the failure, only the spans), and returns the fix in a fixed format.

![Falcon AI fix-with-falcon output for the same trace showing What happened, Root cause in the agent, and a verbatim Current vs Replace with prompt diff](https://fi-cookbook-assets.s3.ap-south-1.amazonaws.com/use-cases/falcon-ai-context-aware-debugging/turn-3-fix-with-falcon.png)

Two things worth noticing about this output. First, Falcon AI quoted the system prompt **verbatim** from the LLM span, not from a guess: the OpenAI auto-instrumentor captured the system message in this run, so the diff is grounded in actual span content. Second, the response references the scorecard from Turn 2: the chat remembered what it discovered two turns ago and used those numbers to predict the impact. That is page context plus conversation memory paying off compounding.

Wall-clock so far: open question to verbatim prompt diff, three turns, about **5 minutes**. None of the turns required you to type a trace ID, paste a span ID, or repeat what the bug was.

## Step 5: Apply the fix and verify with the same query

In [ ]:
SYSTEM_PROMPT = """You are a research assistant for an ML research team. Answer questions using the search_papers tool. Provide citations to support your claims. If search_papers returns no results or an empty list, respond ONLY with: "I could not find any papers in the database matching your query. Please try a different search term." Do NOT use general knowledge, describe papers from memory, or answer without citations."""

# Re-run the exact same failing query
verify = traced_handle(
    user_id="alice",
    session_id="session-bad-verify",
    messages=[{"role": "user", "content": "What are the key papers on contrastive learning for self-supervised vision?"}],
)
print(verify)

# Sanity-check the in-database query still works
ok = traced_handle(
    user_id="alice",
    session_id="session-good-verify",
    messages=[{"role": "user", "content": "What's the seminal paper on transformers?"}],
)
print(ok)

trace_provider.force_flush()

After the fix the contrastive-learning query returns the verbatim refusal, and the transformer query still pulls Vaswani et al. 2017 from the tool result. Both are now grounded in what `search_papers` actually returned.

Open the new traces in the dashboard. The span tree for the contrastive-learning trace now shows the empty tool result followed by the refusal, with no fabricated content in between. Same input, same metric (faithfulness on the citation content), opposite outcome.

Wall-clock from the moment your colleague pinged you to the moment the fix is verified: roughly **8 to 10 minutes**. The hour-long version of this loop (read the spans by hand, write the fix, test, repeat) is the version you do not run today.

Want to ask Falcon AI to confirm the fix worked? Open the new failing-query trace and type "did the fix from the previous trace land?" The chat will compare the two spans and tell you. The page-context awareness carries across traces in the same conversation.

## What you solved

You took a single hallucinated citation, opened Falcon AI directly on the trace, and ran a three-turn conversation that walked from open question to verbatim prompt diff: page context attached the trace automatically, `/analyze-trace-errors` quoted the offending span, and `/fix-with-falcon` returned a paste-ready Current vs Replace with diff. You applied the fix in code, re-ran the same query, and watched the agent refuse instead of fabricate. The manual version of this loop (open spans, scroll, copy out, diff in your head, write the fix, test, repeat) is roughly an hour. The page-context version closes in about ten minutes.

## Next steps

- **Want to lock this fix in as a regression test?** Capture the failing trace as a dataset and run evals against it: see [Building Evaluation Datasets from Production Traces](https://docs.futureagi.com/docs/cookbook/use-cases/falcon-ai-eval-datasets-from-traces).
- **Want the same loop across many traces, not just one?** Run the batch version with `/analyze-trace-errors` on the whole project, then `/build-dataset`, `/run-evaluations`, and `/fix-with-falcon`: see [End-to-End with Falcon AI](https://docs.futureagi.com/docs/cookbook/use-cases/falcon-ai-end-to-end).
- **Need a refresher on tracing setup?** See [Manual Tracing](https://docs.futureagi.com/docs/cookbook/quickstart/manual-tracing) for span decorators, metadata tagging, and prompt template tracking.